<a href="https://colab.research.google.com/github/TetianaMar-888/Amazon_Books_project/blob/main/02_feature_engineering_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Підключення Drive
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_parquet("/content/drive/MyDrive/project_data/books_cleaned.parquet")
print(df.shape)
print(df.columns.tolist())

# 2. Перевіряємо, що всі потрібні колонки на місці
required_cols = ["full_text", "price_clean", "verified_purchase", "helpful_vote",
                  "main_category", "average_rating", "rating_number", "sentiment"]
missing = [c for c in required_cols if c not in df.columns]
print("Відсутні колонки:", missing if missing else "немає, все ок")

# 3. Спліт
from sklearn.model_selection import train_test_split

X = df[["full_text", "price_clean", "verified_purchase", "helpful_vote",
        "main_category", "average_rating", "rating_number"]]
y = df["sentiment"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape, y_train.value_counts(normalize=True).round(3).to_dict())
print("Val:  ", X_val.shape, y_val.value_counts(normalize=True).round(3).to_dict())
print("Test: ", X_test.shape, y_test.value_counts(normalize=True).round(3).to_dict())

import pickle
with open("/content/drive/MyDrive/project_data/train_val_test_split.pkl", "wb") as f:
    pickle.dump((X_train, X_val, X_test, y_train, y_val, y_test), f)

Mounted at /content/drive
(49999, 32)
['rating', 'title_review', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'main_category', 'title_book', 'subtitle', 'author', 'average_rating', 'rating_number', 'features', 'description', 'price', 'store', 'categories', 'details', 'has_image_review', 'has_video', 'has_description', 'price_clean', 'description_clean', 'features_clean', 'full_text', 'sentiment', 'text_length', 'text_clean', 'title_review_clean']
Відсутні колонки: немає, все ок
Train: (34999, 7) {'positive': 0.846, 'neutral': 0.089, 'negative': 0.064}
Val:   (7500, 7) {'positive': 0.846, 'neutral': 0.089, 'negative': 0.064}
Test:  (7500, 7) {'positive': 0.846, 'neutral': 0.089, 'negative': 0.064}


In [ ]:
#TF-IDF для тексту
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),      # враховуємо і окремі слова, і біграми (наприклад "didn't like")
    stop_words="english",
    min_df=5
)

X_train_tfidf = tfidf.fit_transform(X_train["full_text"])
X_val_tfidf = tfidf.transform(X_val["full_text"])
X_test_tfidf = tfidf.transform(X_test["full_text"])

print(X_train_tfidf.shape)

(34999, 10000)


In [ ]:
#Кодування категоріальних/бінарних ознак

import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# verified_purchase вже bool - просто в int
X_train_tab = X_train[["price_clean", "helpful_vote", "average_rating", "rating_number"]].copy()
X_val_tab = X_val[["price_clean", "helpful_vote", "average_rating", "rating_number"]].copy()
X_test_tab = X_test[["price_clean", "helpful_vote", "average_rating", "rating_number"]].copy()

X_train_tab["verified_purchase"] = X_train["verified_purchase"].astype(int)
X_val_tab["verified_purchase"] = X_val["verified_purchase"].astype(int)
X_test_tab["verified_purchase"] = X_test["verified_purchase"].astype(int)

# main_category - one-hot encoding (fit тільки на train!)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_cat = ohe.fit_transform(X_train[["main_category"]])
val_cat = ohe.transform(X_val[["main_category"]])
test_cat = ohe.transform(X_test[["main_category"]])

cat_cols = ohe.get_feature_names_out(["main_category"])
X_train_tab = pd.concat([X_train_tab.reset_index(drop=True),
                          pd.DataFrame(train_cat, columns=cat_cols)], axis=1)
X_val_tab = pd.concat([X_val_tab.reset_index(drop=True),
                        pd.DataFrame(val_cat, columns=cat_cols)], axis=1)
X_test_tab = pd.concat([X_test_tab.reset_index(drop=True),
                         pd.DataFrame(test_cat, columns=cat_cols)], axis=1)

# Масштабування числових ознак (fit тільки на train!)
scaler = StandardScaler()
numeric_cols = ["price_clean", "helpful_vote", "average_rating", "rating_number"]
X_train_tab[numeric_cols] = scaler.fit_transform(X_train_tab[numeric_cols])
X_val_tab[numeric_cols] = scaler.transform(X_val_tab[numeric_cols])
X_test_tab[numeric_cols] = scaler.transform(X_test_tab[numeric_cols])

print(X_train_tab.shape)

(34999, 14)


In [ ]:
# Об'єднання тексту + табличних ознак і збереження
from scipy.sparse import hstack, csr_matrix
import pickle

X_train_full = hstack([X_train_tfidf, csr_matrix(X_train_tab.values)])
X_val_full = hstack([X_val_tfidf, csr_matrix(X_val_tab.values)])
X_test_full = hstack([X_test_tfidf, csr_matrix(X_test_tab.values)])

print(X_train_full.shape)

with open("/content/drive/MyDrive/project_data/features_engineered.pkl", "wb") as f:
    pickle.dump({
        "X_train": X_train_full, "X_val": X_val_full, "X_test": X_test_full,
        "y_train": y_train, "y_val": y_val, "y_test": y_test,
        "tfidf": tfidf, "ohe": ohe, "scaler": scaler
    }, f)

print("Feature engineering завершено і збережено!")

(34999, 10014)
Feature engineering завершено і збережено!
